In [ ]:
!pip install opendatasets

In [ ]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small")


Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: m
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small


100%|██████████| 565M/565M [00:33<00:00, 17.9MB/s]


In [ ]:
cd fashion-product-images-small

/content/fashion-product-images-small


In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv("styles.csv",on_bad_lines='skip')

In [ ]:
df_f=df.copy()

In [ ]:
df_f.head(2)

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans


In [ ]:
df_f['image']=df_f.apply(lambda row:str(row['id'])+'.jpg',axis=1)

In [ ]:
df_f.head(2)

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt,15970.jpg
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans,39386.jpg


In [ ]:
df_f=df_f.sample(frac=1).reset_index(drop=True)
df_f.head(2)

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image
0,2130,Men,Apparel,Topwear,Shirts,Green,Spring,2011.0,Casual,Basics Men Green & White Slim Fit Shirt,2130.jpg
1,58860,Men,Accessories,Cufflinks,Cufflinks,Copper,Summer,2012.0,Casual,Hakashi Men Copper Cufflinks,58860.jpg


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split

In [ ]:
batch_size = 256

# Image transformation
image_transforms = transforms.Compose([
    transforms.Resize((60, 80)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Custom Dataset Class
class FashionDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df_f = dataframe
        self.img_dir = img_dir
        self.transform = transform

        self.df_f['label_code'] = self.df_f['masterCategory'].astype('category').cat.codes
        self.class_indices = dict(enumerate(self.df_f['masterCategory'].astype("category").cat.categories))
        self.class_indices = {v: k for k, v in self.class_indices.items()}

    def __len__(self):
        return len(self.df_f)

    def __getitem__(self, idx):
        image_name = self.df_f.iloc[idx]['image']
        label = self.df_f.iloc[idx]['label_code']
        img_path = os.path.join(self.img_dir, image_name)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)



df_f['file_exists'] = df_f['image'].apply(lambda x: os.path.exists(os.path.join("images", x)))


missing_count = len(df_f) - df_f['file_exists'].sum()
print(f"Missing photos numbers: {missing_count}")

df_f = df_f[df_f['file_exists'] == True].reset_index(drop=True)
df_f = df_f.drop(columns=['file_exists'])

full_dataset = FashionDataset(dataframe=df_f, img_dir="images", transform=image_transforms)



Missing photos numbers: 5


In [ ]:
# Splitting the data
val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Dataloader
training_generator = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_generator = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

classes = len(full_dataset.class_indices)
print(f"The Total classes: {classes}")
print(f"Class Indices: {full_dataset.class_indices}")

The Total classes: 7
Class Indices: {'Accessories': 0, 'Apparel': 1, 'Footwear': 2, 'Free Items': 3, 'Home': 4, 'Personal Care': 5, 'Sporting Goods': 6}


In [ ]:
# Model Architecture
class Fashion_CNN(nn.Module):
    def __init__(self):
        super(Fashion_CNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=(3, 3), padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2))

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3, 3), padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2))

        dummy_input = torch.zeros(1, 3, 60, 80)

        with torch.no_grad():
            dummy_features = self.pool1(F.relu(self.conv1(dummy_input)))
            dummy_features = self.pool2(F.relu(self.conv2(dummy_features)))
            num_features_after_conv = dummy_features.numel()

        print(f"Linear input size: {num_features_after_conv}")

        self.fc1 = nn.Linear(in_features=num_features_after_conv, out_features=256)
        self.fc2 = nn.Linear(in_features=256, out_features=128)
        self.fc3 = nn.Linear(in_features=128, out_features=64)
        self.fc_out = nn.Linear(in_features=64, out_features=classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc_out(x)
        return x

In [ ]:
# Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')

model = Fashion_CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Main Training and Evaluate phase
def train_and_evaluate(model, training_generator, validation_generator, epochs=10):
    print(f"Starting training on device {device}\n" + "-"*40)

    for epoch in range(epochs):
        # ------------------------------------------
        #               TRAINING PHASE
        # ------------------------------------------
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for images, labels in training_generator:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        epoch_train_loss = running_loss / len(training_generator.dataset)
        epoch_train_acc = (correct_train / total_train) * 100

        # ------------------------------------------
        #             VALIDATION PHASE
        # ------------------------------------------
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for images, labels in validation_generator:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        epoch_val_loss = val_loss / len(validation_generator.dataset)
        epoch_val_acc = (correct_val / total_val) * 100

        print(f"Epoch [{epoch+1}/{epochs}]")
        print(f"  Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")
        print(f"  Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.2f}%")
        print("-" * 40)


train_and_evaluate(model, training_generator, validation_generator, epochs=10)

Linear input size: 19200
Starting training on device cuda
----------------------------------------
Epoch [1/10]
  Train Loss: 0.3660 | Train Acc: 87.30%
  Val Loss:   0.1810 | Val Acc:   94.13%
----------------------------------------
Epoch [2/10]
  Train Loss: 0.1321 | Train Acc: 96.04%
  Val Loss:   0.1214 | Val Acc:   96.60%
----------------------------------------
Epoch [3/10]
  Train Loss: 0.0945 | Train Acc: 97.21%
  Val Loss:   0.0973 | Val Acc:   97.26%
----------------------------------------
Epoch [4/10]
  Train Loss: 0.0746 | Train Acc: 97.81%
  Val Loss:   0.0859 | Val Acc:   97.65%
----------------------------------------
Epoch [5/10]
  Train Loss: 0.0556 | Train Acc: 98.43%
  Val Loss:   0.0851 | Val Acc:   97.92%
----------------------------------------
Epoch [6/10]
  Train Loss: 0.0441 | Train Acc: 98.83%
  Val Loss:   0.0935 | Val Acc:   97.51%
----------------------------------------
Epoch [7/10]
  Train Loss: 0.0352 | Train Acc: 98.99%
  Val Loss:   0.0910 | Val Acc:

In [ ]:
# Model Weight Save
torch.save(model.state_dict(), 'fashion_cnn_98_acc.pth')
print(" Model saved successfully! ")

 Model saved successfully! 
